In [1]:
#%%

import typing as T
import pathlib as P
import json
import pickle
import sys
import os
prj_root = P.Path("__file__").absolute().parent.parent.parent
if (p := str(prj_root)) not in sys.path:
    sys.path.append(p)
import collections as clt
import itertools as it
import functools as ft
import operator as opr

In [2]:
#%%

from dataset.hg_dataset import DBLPDataset
from models.HGAT import HGAT
import util.metrics as um

/data0/shaojiangyi/anaconda3/envs/py311_torch240/lib/python3.11/site-packages/dgl/heterograph_index.py:8: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/data0/shaojiangyi/anaconda3/envs/py311_torch240/lib/python3.11/site-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()
/data0/shaojiangyi/anaconda3/envs/py311_torch240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonote

In [3]:
#%%

import torch
import torch as th
import numpy as np
import dgl
from tqdm import tqdm
from torch.cuda.amp import autocast
import torch.nn.functional as F

In [4]:
#%%

ns = ["cc", "mf", "bp"]
ontology_lst = ["cellular_component", "molecular_function", "biological_process"]

In [5]:
dt_root = P.Path("./data")
def get_predictions_by_split(split: str = "test", bs: int = 64):
    """
    get predictions for a given data split ('train', 'valid', or 'test').

    Args:
        split (str): Which split to get predictions from. One of 'train', 'valid', 'test'.
        bs (int): Batch size.

    Returns:
        List[Dict[str, torch.Tensor]]: List of dicts mapping protein names to feature tensors for each ontology.
    """
    feats_lst = []
    preds_lst = []
    testids_lst = []
    for x in ns:
        device = th.device("cuda" if th.cuda.is_available() else "cpu")
        dataset = DBLPDataset(dt_root, x, 'HGAT', bs, device)

        g = dataset.g.to(device)

        model = HGAT(dataset.node_type, num_classes=dataset.go_num, feature_dim=dataset.feature_dim, hidden_dim=128, num_layers=2)

        # load checkpoint
        ckpt_root = prj_root / "models"
        ckpt_path = ckpt_root / f"hgat_{x}_best"
        state_dict = th.load(ckpt_path)
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval()

        # Select loader based on split
        if split == "train":
            loader = dataset.train_loader
        elif split == "valid":
            loader = dataset.valid_loader
        elif split == "test":
            loader = dataset.test_loader
        else:
            raise ValueError(f"Unknown split: {split}")

        loader_tqdm = tqdm(loader, ncols=120)

        y_trues = []
        y_predicts = []
        prot_indices = []
        with torch.no_grad():
            for i, (sample_nodes, seed, blocks) in enumerate(loader_tqdm):
                with autocast():
                    seed = seed['protein']
                    subgraph = dgl.node_subgraph(g, sample_nodes)
                    label = dataset.get_label(seed.cpu().numpy().tolist())
                    h_dict = subgraph.ndata['h']
                    pred = model(subgraph, h_dict)

                    seed_indices = [torch.where(sample_nodes['protein'] == x)[0].item() for x in seed]

                    pred = pred['protein'][seed_indices]
                    label = torch.tensor(label).to(device)
                    label = label[seed_indices]
                    prot_indices += seed.cpu().numpy().tolist()

                    if torch.isnan(pred).any():
                        print(pred)
                        raise ValueError("nan")

                    pred = F.sigmoid(pred)
                    y_trues.append(label.to(torch.float32).detach().cpu())
                    y_predicts.append(pred.to(torch.float32).detach().cpu())
            y_trues = torch.cat(y_trues, dim=0)
            y_predicts = torch.cat(y_predicts, dim=0)

        # evaluate
        fmax, aupr = um.fmax_score(y_trues, 
                                   y_predicts, auprc=True)
        print(f'Ontology: {x} | fmax: {fmax}, aupr: {aupr}')

        testids_lst.append(prot_indices)
        # print(y_trues.shape, y_predicts.shape)
        preds_lst.append(torch.stack([y_trues, y_predicts]).transpose(0,1))

    # get protein name id
    name_path = prj_root / "data" / "protein_name.txt"
    with open(name_path, "r") as f:
        prot_lst = f.read().splitlines()

    # build a dict for protein name to features'
    protpred = [{prot_lst[j]: i for j, i in zip(testids, preds)}
                      for testids, preds in zip(testids_lst, preds_lst)]
    return protpred

In [6]:
#%%

# get feature for each split
split_type = ["train", "valid", "test"]
pred_states = [get_predictions_by_split(x)
                for x in split_type]

100%|███████████████████████████████████████████████████████████████████████████████| 1547/1547 [01:19<00:00, 19.58it/s]


Ontology: cc | fmax: 0.5646084107459886, aupr: 0.5660719165867558


100%|███████████████████████████████████████████████████████████████████████████████| 1544/1544 [01:53<00:00, 13.56it/s]


Ontology: mf | fmax: 0.48359631640101847, aupr: 0.4320308738097129


/data0/shaojiangyi/pprogo-flg-1/dataset/hg_dataset.py:235: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  node = pd.read_csv(data_path.joinpath('node.dat'), header=None, sep='\t')
100%|███████████████████████████████████████████████████████████████████████████████| 1544/1544 [04:09<00:00,  6.20it/s]


...

Ontology: cc | fmax: 0.6204641021154815, aupr: 0.6458389155492223


100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 13.11it/s]


Ontology: mf | fmax: 0.6399903287701163, aupr: 0.6236617850206021


/data0/shaojiangyi/pprogo-flg-1/dataset/hg_dataset.py:235: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  node = pd.read_csv(data_path.joinpath('node.dat'), header=None, sep='\t')
100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:01<00:00,  6.38it/s]


Ontology: bp | fmax: 0.37335882722858926, aupr: 0.30832170639463763


In [9]:
# saving features
saving_dir = prj_root / "data" / "prediction"
# feat_state = dict(zip(ontology_lst, featdict_lst))
# saving_path = saving_dir / "hgat_features.pt"
for i,t in enumerate(split_type):
    saving_path = saving_dir / f"hgat_predictions_{t}.pt"
    th.save(pred_states[i], saving_path)